# RAG Demonstration

In [1]:
# Importing packages
import pandas as pd
import torch
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, logging, AutoModel
logging.set_verbosity_error()
import random
import re
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
from typing import List
from dotenv import load_dotenv
import os
from torch import Tensor
import faiss 
import json

In [2]:
# Loading token
load_dotenv('token.env')
token = os.getenv('HUGGINGFACE_TOKEN')

# Loading model - pass token directly
model_name = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, token=token)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto",
    token=token  # Pass token here
)

In [3]:
df = pd.read_csv("/work/mbouthil/projects/research_project/MEDRAG/synthetic_data/synq.csv")
queries = df['QUERY'].tolist()
passages = df['PASSAGE'].tolist()
# df.head(20)

### Example of a Query:

In [4]:
# print(queries[0])

### Example Passage:

In [5]:
# print(passages[0])

### For Patient:

In [6]:
print('Subject ID: ' + str(df['SUBJECT_ID'].iloc[0]))

Subject ID: 22532


### Loading the vector DB, index and json

In [7]:
# Loading Index
index = faiss.read_index("/work/mbouthil/projects/research_project/MEDRAG/retrieval_data/passage.index")

# Load Metadata
metadata = []
with open("/work/mbouthil/projects/research_project/MEDRAG/retrieval_data/passage_metadata.jsonl") as f:
    for line in f:
        metadata.append(json.loads(line))

### Loading the Trained Query Encoder

In [8]:
# Loading Query Encoder
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
query_encoder = AutoModel.from_pretrained("bert-base-uncased")

query_encoder = AutoModel.from_pretrained(
    "/work/mbouthil/projects/research_project/MEDRAG/model_weights/query_encoder"
) # .to("cuda")

query_encoder.eval()

def encode_query(query:str, batch_size:int=32) -> Tensor:

    embeddings = []

    with torch.no_grad():
        inputs = tokenizer(
            query, 
            padding=True,
            truncation=True,
            return_tensors="pt",
            max_length=512
        ) #.to("cuda")

    outputs = query_encoder(**inputs)
    cls_embeddings = outputs.last_hidden_state[:, 0] 

    embeddings.append(cls_embeddings.cpu())

    return torch.cat(embeddings, 0)

### Testing custom question

In [9]:
question = 'Subject ID: ' + str(df['SUBJECT_ID'].iloc[0]) + '\n'  +  "Does the patient have a history of tuberculosis?"
print(question)

Subject ID: 22532
Does the patient have a history of tuberculosis?


In [10]:
# Embedding Query
query_emb = encode_query([question]).detach().cpu().numpy()

In [21]:
# Matching for the top K=5 highest scores
K = 10
scores, ids = index.search(query_emb, K)

In [22]:
print(scores)

[[60.6493   59.087296 58.393707 57.691124 57.28512  57.28512  56.782654
  56.731564 56.72872  56.69512 ]]


In [24]:
# candidates = [metadata[i]["text"] for i in ids[0]]
# for i, passage in enumerate(candidates):
#     print(scores[0][i])
#     print("\n\n\n")

# Other infromation

### At this time:

In [13]:
# Synthetic Dataset length:
print(len(df))

28431


In [14]:
# Additional Synthetic Dataset Length:
add_df = pd.read_csv("/work/mbouthil/projects/research_project/MEDRAG/synthetic_data/add_synq.csv")
print(len(add_df))

79825


In [15]:
# add_df.head(20)